# DNN — Strategy Discovery

Find the optimal scaling-in strategy for the DNN (Residual MLP) model using the shared `strategy_engine` module.

**Method:** Grid search + walk-forward evaluation (5 folds) on per-snapshot predictions. Model is loaded from exported artifacts (trained in `02_export.ipynb`). Strategy selected by best mean Sharpe ratio across folds.

**Output:** `data/optimal_strategy_dnn.json`

In [1]:
import sys

sys.path.insert(0, str(__import__("pathlib").Path.cwd().parent))

import json
from datetime import UTC, datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from strategy_engine import StrategyGrid, WalkForwardEvaluator

np.random.seed(42)

MAX_BID = 0.85

## 1. Load DNN Model & Data

In [2]:
# Load features
rows = []
with open("../../data/latest_features.jsonl") as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)
df["target"] = (df["outcome"] == "UP").astype(int)
print(f"Total snapshots: {len(df):,} | Candles: {df['candle_id'].nunique():,}")

Total snapshots: 245,319 | Candles: 5,187


In [3]:
# Define model classes (needed for torch.load to unpickle)
class ResidualBlock(torch.nn.Module):
    def __init__(self, hidden_size, dropout=0.3):
        super().__init__()
        self.block = torch.nn.Sequential(
            torch.nn.Linear(hidden_size, hidden_size),
            torch.nn.LayerNorm(hidden_size),
            torch.nn.GELU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(hidden_size, hidden_size),
            torch.nn.LayerNorm(hidden_size),
        )
        self.act = torch.nn.GELU()

    def forward(self, x):
        return self.act(x + self.block(x))


class ResidualMLP(torch.nn.Module):
    def __init__(self, input_size=11, hidden=128, n_blocks=4, dropout=0.3):
        super().__init__()
        self.input_proj = torch.nn.Sequential(
            torch.nn.Linear(input_size, hidden),
            torch.nn.LayerNorm(hidden),
            torch.nn.GELU(),
            torch.nn.Dropout(dropout),
        )
        self.blocks = torch.nn.ModuleList([ResidualBlock(hidden, dropout) for _ in range(n_blocks)])
        self.head = torch.nn.Linear(hidden, 1)

    def forward(self, x):
        x = self.input_proj(x)
        for block in self.blocks:
            x = block(x)
        return self.head(x)


# Load exported DNN model
model = torch.load("../../models/dnn_v1.pt", weights_only=False)
model.eval()
scaler = joblib.load("../../models/dnn_scaler_v1.joblib")
feature_cols = joblib.load("../../models/dnn_feature_cols_v1.joblib")
print(f"DNN model loaded: {len(feature_cols)} features")

DNN model loaded: 11 features


## 2. Build Per-Snapshot Predictions

Generate predictions for each snapshot using the DNN model. The model uses raw features (single-snapshot, no temporal buffering).

In [4]:
candle_data = []

for cid in df["candle_id"].unique():
    snap_rows = df[df["candle_id"] == cid].sort_values("timestamp")
    if len(snap_rows) < 5:
        continue

    truth = int(snap_rows["target"].iloc[0])
    X = snap_rows[feature_cols].fillna(0).values.astype(np.float32)
    X_scaled = scaler.transform(X)

    with torch.no_grad():
        logits = model(torch.tensor(X_scaled, dtype=torch.float32))
        probs = torch.sigmoid(logits).numpy().flatten()

    up_asks = snap_rows["up_best_ask"].values
    down_asks = snap_rows["down_best_ask"].values
    elapsed = snap_rows["elapsed_pct"].values

    snapshots = []
    for i in range(len(snap_rows)):
        ua = float(up_asks[i]) if up_asks[i] is not None and np.isfinite(up_asks[i]) else None
        da = float(down_asks[i]) if down_asks[i] is not None and np.isfinite(down_asks[i]) else None
        snapshots.append(
            {
                "tick": i,
                "elapsed_pct": float(elapsed[i]),
                "pred": int(probs[i] >= 0.5),
                "prob": float(probs[i]),
                "up_ask": ua,
                "down_ask": da,
            }
        )

    candle_data.append({"candle_id": cid, "truth": truth, "snapshots": snapshots})

print(f"Built predictions for {len(candle_data)} candles")

Built predictions for 5187 candles


## 3. Strategy Grid Search + Walk-Forward Evaluation

Use the shared `strategy_engine` module for grid search (~3,800 combinations) with 5-fold walk-forward cross-validation.

In [5]:
grid = StrategyGrid()
strategies = grid.generate()
print(f"Grid size: {len(strategies):,} strategy combinations")

evaluator = WalkForwardEvaluator(
    strategies=strategies,
    candle_data=candle_data,
    n_folds=5,
    max_bid=MAX_BID,
)
results_df = evaluator.run()
print("\nTop 10 strategies by mean Sharpe:")
print(
    results_df[["strategy", "sharpe_mean", "sharpe_std", "win_rate_mean", "return_mean", "total_bets_mean"]]
    .head(10)
    .to_string(index=False)
)

Grid size: 3,815 strategy combinations



Top 10 strategies by mean Sharpe:
                  strategy  sharpe_mean  sharpe_std  win_rate_mean  return_mean  total_bets_mean
       1x e40%c4 conf>0.70     0.036235    0.040255       0.869841     0.799358              7.2
       1x e45%c4 conf>0.70     0.036235    0.040255       0.869841     0.799358              7.2
       1x e35%c4 conf>0.70     0.036235    0.040255       0.869841     0.799358              7.2
       1x e50%c4 conf>0.70     0.036235    0.040255       0.869841     0.799358              7.2
2x e35%c4+e50%c4 conf>0.70     0.036235    0.040255       0.869841     1.598716             14.4
2x e35%c4+e45%c4 conf>0.70     0.036235    0.040255       0.869841     1.598716             14.4
2x e15%c4+e35%c4 conf>0.70     0.034178    0.040017       0.860485     1.398716             14.6
2x e10%c4+e35%c4 conf>0.70     0.034178    0.040017       0.860485     1.398716             14.6
2x e20%c4+e45%c4 conf>0.70     0.034178    0.040017       0.860485     1.398716             

## 4. Export Best Strategy

In [ ]:
from polybot.domain.trading_strategy import TradingStrategy  # noqa: E402

best_row = results_df.iloc[0]
best_cfg = best_row["_config"]

config = {
    "model": "dnn",
    "strategy": best_row["strategy"],
    "entry_points": best_cfg.entry_points,
    "min_confidence": best_cfg.min_confidence,
    "sharpe_mean": round(best_row["sharpe_mean"], 4),
    "sharpe_std": round(best_row["sharpe_std"], 4),
    "win_rate": round(best_row["win_rate_mean"], 4),
    "return_pct": round(best_row["return_mean"], 2),
    "max_drawdown": round(best_row["max_dd_mean"], 4),
    "n_folds": 5,
    "grid_size": len(strategies),
    "eval_candles": len(candle_data),
    "eval_method": "walk_forward_5_folds",
    "total_bets": int(best_row["total_bets_mean"]),
    "created_at": datetime.now(UTC).isoformat(),
}

out_path = Path("../../data/optimal_strategy_dnn.json")
with open(out_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"Best strategy: {best_row['strategy']}")
print(f"  Entry points: {best_cfg.entry_points}")
print(f"  Min confidence: {best_cfg.min_confidence}")
print(f"  Sharpe: {best_row['sharpe_mean']:.4f} +/- {best_row['sharpe_std']:.4f}")
print(f"  Win rate: {best_row['win_rate_mean'] * 100:.1f}%")
print(f"  Return: {best_row['return_mean']:+.1f}%")
print(f"  Max DD: {best_row['max_dd_mean'] * 100:.1f}%")
print(f"\nSaved to {out_path}")

ts = TradingStrategy.from_json(str(out_path), name="DNN")
print(f"\nTradingStrategy loaded: {ts.name}, entry_points={ts.entry_points}, min_confidence={ts.min_confidence}")